# 2차 파인튜닝 — 합성 5,883장 + 실전 수확 354장 혼합

**1차와 차이:** 대림 실사진에서 파이프라인이 카탈로그로 검증한 라벨 크롭(실제 광학 열화,
제목복구 hard 샘플 74장 포함)을 학습에 추가. 평가는 **실전 val**로 해서 실전 성능 기준으로 베스트를 고름.

**업로드 파일:** `synth_rec.zip` + `real_rec_data.zip`

사용법: GPU(T4) 런타임 → 셀 순서대로. (셀1 후 **세션 다시 시작** 필수)

In [ ]:
# 1) 설치 — torch 제거(NCCL 충돌 방지) 후 GPU paddle
!pip uninstall -y -q torch torchvision torchaudio 2>/dev/null
!pip install -q paddlepaddle-gpu==3.0.0 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!git clone --depth 1 https://github.com/PaddlePaddle/PaddleOCR.git
!pip install -q -r PaddleOCR/requirements.txt
print('✅ 설치 완료 — [런타임 → 세션 다시 시작] 후 셀2부터!')

In [ ]:
# 2) 데이터 업로드(2개) + 배치 + 라벨 병합 (실전은 8배 오버샘플 — 5,883 대 319 불균형 보정)
from google.colab import files
up = files.upload()   # synth_rec.zip, real_rec_data.zip 둘 다 선택
!unzip -oq synth_rec.zip -d PaddleOCR/train_data/
!unzip -oq real_rec_data.zip -d PaddleOCR/train_data/
import io
def read(p): return io.open(p, encoding='utf-8').read().splitlines()
synth_tr = [f'synth_rec/train/{l.split(chr(9))[0]}\t{l.split(chr(9))[1]}' for l in read('PaddleOCR/train_data/synth_rec/train/rec_gt_train.txt')]
real_tr  = [f'real_rec_data/{l.split(chr(9))[0]}\t{l.split(chr(9))[1]}' for l in read('PaddleOCR/train_data/real_rec_data/train.txt')]
real_va  = [f'real_rec_data/{l.split(chr(9))[0]}\t{l.split(chr(9))[1]}' for l in read('PaddleOCR/train_data/real_rec_data/val.txt')]
merged = synth_tr + real_tr * 8
io.open('PaddleOCR/train_data/train_merged.txt', 'w', encoding='utf-8').write('\n'.join(merged) + '\n')
io.open('PaddleOCR/train_data/val_real.txt', 'w', encoding='utf-8').write('\n'.join(real_va) + '\n')
print(f'train {len(merged)} (합성 {len(synth_tr)} + 실전 {len(real_tr)}x8) · val(실전) {len(real_va)}')

In [ ]:
# 3) config·사전학습 모델 자동 탐색 (1차와 동일)
%cd /content
import glob, os
cfgs = glob.glob('PaddleOCR/configs/rec/**/*korean*', recursive=True)
CFG = next((c for c in cfgs if 'v5' in c.lower() and 'mobile' in c.lower()), cfgs[0] if cfgs else None)
print('사용 config:', CFG)
urls = [
 'https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/korean_PP-OCRv5_mobile_rec_pretrained.pdparams',
 'https://paddleocr.bj.bcebos.com/PP-OCRv5/multilingual/korean_PP-OCRv5_mobile_rec_pretrained.pdparams',
]
for u in urls:
    if os.system(f'wget -q {u} -O pretrain.pdparams') == 0 and os.path.getsize('pretrain.pdparams') > 1e6:
        print('사전학습 확보:', u); break

In [ ]:
# 4) 학습 (20 epochs, T4 기준 ~60-80분)
# 주의: 이 config의 실제 배치는 Train.sampler.first_bs가 결정 (MultiScaleSampler)
%cd /content/PaddleOCR
CFG_REL = CFG.split('PaddleOCR/')[1] if CFG.startswith('PaddleOCR/') else CFG
!python tools/train.py -c {CFG_REL} \
  -o Global.pretrained_model=/content/pretrain \
     Global.epoch_num=20 \
     Global.save_model_dir=./output/korean_lowres_v2 \
     Global.eval_batch_step="[0,500]" \
     Optimizer.lr.learning_rate=0.0001 \
     Train.sampler.first_bs=32 \
     Train.dataset.data_dir=./train_data \
     Train.dataset.label_file_list=["./train_data/train_merged.txt"] \
     Eval.dataset.data_dir=./train_data \
     Eval.dataset.label_file_list=["./train_data/val_real.txt"]

In [ ]:
# 5) 추론 모델로 내보내기 + 다운로드
%cd /content/PaddleOCR
CFG_REL = CFG.split('PaddleOCR/')[1] if CFG.startswith('PaddleOCR/') else CFG
!python tools/export_model.py -c {CFG_REL} \
  -o Global.pretrained_model=./output/korean_lowres_v2/best_accuracy \
     Global.save_inference_dir=./korean_lowres_v2_rec_infer
!zip -q -r /content/korean_lowres_v2_rec_infer.zip korean_lowres_v2_rec_infer
from google.colab import files
files.download('/content/korean_lowres_v2_rec_infer.zip')
print('로컬 A/B: daelim_closeup.py --rec_dir korean_lowres_v2_rec_infer 로 1차 모델과 비교')